#### Data loading

In [1]:
import pandas as pd
import numpy as np
import re
import plotly.express as px
from collections import Counter
from cleantext import clean

In [2]:
from datasets import load_dataset
ds = load_dataset("go_emotions", split="train")

In [ ]:
print(ds[20])

In [ ]:
label_names = ds.features["labels"].feature.names
print(label_names)

df = ds.to_pandas()
df.head()

In [ ]:
df.drop(columns=["id"], inplace=True)
df.head()

#### Data Cleaning

In [ ]:
def clean_text(text):
    # 1. Use cleantext to fix Unicode + transliterate
    text = clean(text,
                 fix_unicode=True,
                 to_ascii=True,
                 lower=False,
                 no_emoji=True,
                 no_urls=True,
                 no_currency_symbols=False)

    # 2. Remove stray currency symbols NOT followed by a number
    text = re.sub(r'([$€£₹])(?!\d)', '', text)

    # 3. Remove unwanted punctuation excluding . , ! ? $ % &
    text = re.sub(r'[{}\[\]<>@#*`~/\\\^_|+=\-—…;:\(\)]', '', text)

    # 5. Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['text'] = df['text'].apply(clean_text)
df.head()


In [ ]:
all_labels = [label for sublist in df['labels'] for label in sublist]
counts = Counter(all_labels)

df_counts = pd.DataFrame({
    'label_index': list(range(len(label_names))),
    'label': label_names,
    'count': [counts.get(i, 0) for i in range(len(label_names))]
})

df_counts = df_counts.sort_values('count', ascending=False).reset_index(drop=True)
df_counts["count_display"] = df_counts["count"].clip(upper=6000)
fig = px.bar(
    df_counts,
    y='label',
    x='count_display',
    text='count',
    color="label",
)

fig.update_traces(marker_line_color='white', textposition='outside', textfont=dict(size=8))

fig.update_layout(bargap=0.15, plot_bgcolor="white", height=700, width=800,
                  showlegend=False,yaxis_tickfont_size=9, xaxis_tickfont_size=9,
                  xaxis_title='', yaxis_title=''
)
fig.update_xaxes(showticklabels=False) 
fig.show()

In [ ]:
type(df["labels"].loc[27])

In [ ]:
df.info()  

#### If you want to balance the dataset, you can use the following techniques:

In [ ]:
single_label_df = df[df['labels'].apply(lambda x: len(x) == 1)].copy()
single_label_df['class'] = single_label_df['labels'].apply(lambda x: x[0])
counts = Counter(single_label_df['class'])
multi_label_df = df[df['labels'].apply(lambda x: len(x) > 1)].copy()

In [ ]:
neutral_idx = 27

# Single-label: only neutral
neutral_single = single_label_df[single_label_df['class'] == neutral_idx]
neutral_single_count = len(neutral_single)

# Multi-label: neutral present in multi-label samples
neutral_multi = multi_label_df[multi_label_df['labels'].apply(lambda x: neutral_idx in x)]
neutral_multi_count = len(neutral_multi)

import plotly.graph_objects as go

# Bar plot for counts
fig1 = go.Figure(data=[
    go.Bar(
        x=['Single-label Neutral', 'Multi-label Neutral'],
        y=[neutral_single_count, neutral_multi_count],
        text=[neutral_single_count, neutral_multi_count],
        textposition='outside',
        marker_color=['#636efa', '#EF553B']
    )
])
fig1.update_layout(
    title='Neutral Label Occurrences',
    yaxis_title='Count',
    xaxis_title='Type',
    showlegend=False,
    height=600,
    width=800, title_x=0.5
)
fig1.show()

# Pie chart for proportions
fig2 = go.Figure(data=[
    go.Pie(
        labels=['Single-label Neutral', 'Multi-label Neutral'],
        values=[neutral_single_count, neutral_multi_count],
        textinfo='label+percent',
        marker=dict(colors=['#636efa', '#EF553B'])
    )
])
fig2.update_layout(title='Neutral Label Distribution', height=400, width=600)
fig2.show()

In [ ]:
thresh = 6000
# Downsample each class to at most thresh samples
balanced_single_label_df = (
    single_label_df.groupby('class', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), thresh), random_state=42), include_groups=False)
    .reset_index(drop=True)
)

# Keep all multi-label samples
multi_label_df = df[df['labels'].apply(lambda x: len(x) > 1)].copy()

# Combine balanced single-label and all multi-label samples
df_balanced = pd.concat([balanced_single_label_df[['text', 'labels']], multi_label_df[['text', 'labels']]], ignore_index=True)
print(f"Original dataset size: {len(df)}")
print(f"Balanced dataset size: {len(df_balanced)}")
df_balanced.head()

In [ ]:
df_balanced.info()

In [ ]:
all_labels_balanced = [label for sublist in df_balanced['labels'] for label in sublist]
counts = Counter(all_labels_balanced)

df_counts_balanced = pd.DataFrame({
    'label_index': list(range(len(label_names))),
    'label': label_names,
    'count': [counts.get(i, 0) for i in range(len(label_names))]
})

df_counts_balanced = df_counts_balanced.sort_values('count', ascending=False).reset_index(drop=True)

fig = px.bar(df_counts_balanced, x='label', y='count', text='count',
             labels={'label': 'Label', 'count': 'Count'}, 
             color= "label",
             title= 'Go-Emotions Label Distribution in Balanced Training Set',
        )

fig.update_traces(marker_line_width=1, marker_line_color='white', textposition='outside')
fig.update_layout(bargap=0.15, xaxis_tickangle=-60, margin=dict(t=60, b=160),height=500, width=1200, showlegend=False, title_x=0.5)
fig.show()

In [ ]:
df_balanced_shuffled = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
df_balanced_shuffled.head()

#### Data Preparation for Model

In [ ]:
"IF YOU WANT TO USE BALANCED DATASET"
def parse_labels(x):
    if isinstance(x, (list, np.ndarray)):
        return list(map(int, x))
    elif isinstance(x, float) and pd.isna(x):
        return []
    else:
        x = x.strip("[]")
        return list(map(int, x.split())) if x else []

df_balanced_shuffled['labels'] = df_balanced_shuffled['labels'].apply(parse_labels)

In [ ]:
"O/W:"
def parse_labels(x):
    if isinstance(x, (list, np.ndarray)):
        return list(map(int, x))
    elif isinstance(x, float) and pd.isna(x):
        return []
    else:
        x = x.strip("[]")
        return list(map(int, x.split())) if x else []
    
df['labels'] = df['labels'].apply(parse_labels)

In [ ]:
"IF YOU WANT TO USE BALANCED DATASET"
print(type(df_balanced_shuffled['labels'][0]))

In [ ]:
"O/W:"
print(type(df['labels'][0]))

In [ ]:
"IF YOU WANT TO USE BALANCED DATASET"
one_hot = pd.get_dummies(df_balanced_shuffled['labels'].explode()).groupby(level=0).max().fillna(0).astype(int)
df1 = pd.concat([df_balanced_shuffled, one_hot], axis=1)
df1.head()

In [ ]:
"O/W:"
one_hot = pd.get_dummies(df['labels'].explode()).groupby(level=0).max().fillna(0).astype(int)
df1 = pd.concat([df, one_hot], axis=1)
df1.head()

In [ ]:
df1.drop(columns=['labels'], inplace=True)
df1.head()

In [ ]:
df1.info()

In [ ]:
df1.to_csv(r"Go-Emotions-Train.csv", index=False)